# Panthera-HT digital twin: train ACT and SmolVLA on Kaggle

Input: the raw episodes recorded on Ubuntu with `scripts/linux/twin_record.py`, zipped and uploaded as a
Kaggle Dataset (the folder contains `episode_*/`, `lerobot_convert.py` and `policy_bridge.py`).

Settings: **Accelerator = GPU T4 x2** (one GPU is used), **Internet = On** (downloads pretrained weights).
Tested pipeline: conversion and a 3-step ACT run with lerobot 0.4.4 (CPU). SmolVLA has not been run yet
(no GPU/Hugging Face access where it was prepared); the T4 has no native bfloat16, so check the first log lines.

In [ ]:
# 1) Install LeRobot (same version the converter was tested with) and check the GPU.
!pip install -q "lerobot[smolvla]==0.4.4"
import torch, subprocess
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2) Paths: change RAW to your uploaded dataset folder.
from pathlib import Path
import glob
candidates = [Path(p).parent for p in glob.glob("/kaggle/input/**/lerobot_convert.py", recursive=True)]
print("found raw folders:", candidates)
RAW = candidates[0]
DATA = Path("/kaggle/working/panthera_twin")
REPO = "local/panthera_twin"
print("episodes:", len(list(RAW.glob("episode_*"))))

In [ ]:
# 3) Convert raw episodes -> LeRobot dataset (videos). Use --vcodec h264 if the default codec fails.
!rm -rf {DATA}
!python {RAW}/lerobot_convert.py --raw {RAW} --root {DATA} --repo-id {REPO}

## ACT baseline (no language; the policy used by the Panthera-HT_lerobot repo)

In [ ]:
ACT_OUT = "/kaggle/working/train_act"
!lerobot-train --dataset.repo_id={REPO} --dataset.root={DATA} \
  --policy.type=act --policy.device=cuda --policy.push_to_hub=false \
  --batch_size=8 --steps=30000 --save_freq=10000 --log_freq=200 \
  --num_workers=2 --output_dir={ACT_OUT} --job_name=panthera_twin_act --wandb.enable=false

## SmolVLA fine-tune (language-conditioned)
`smolvla_base` expects `camera1..3`: wrist -> camera1, workspace -> camera2, camera3 left empty.

In [ ]:
import json
RENAME = json.dumps({"observation.images.wrist": "observation.images.camera1",
                     "observation.images.workspace": "observation.images.camera2"})
SMOLVLA_OUT = "/kaggle/working/train_smolvla"
!lerobot-train --dataset.repo_id={REPO} --dataset.root={DATA} \
  --policy.path=lerobot/smolvla_base --policy.device=cuda --policy.push_to_hub=false \
  --policy.empty_cameras=1 --rename_map='{RENAME}' \
  --batch_size=16 --steps=20000 --save_freq=5000 --log_freq=200 \
  --num_workers=2 --output_dir={SMOLVLA_OUT} --job_name=panthera_twin_smolvla --wandb.enable=false

## Package the checkpoints for download (Output tab)

In [ ]:
!cd /kaggle/working && for d in train_act train_smolvla; do \
  [ -d $d/checkpoints/last/pretrained_model ] && tar czf ${d}_last.tgz -C $d/checkpoints/last pretrained_model; done; ls -lh /kaggle/working/*.tgz

## Run a checkpoint against the twin (on the Ubuntu laptop)
1. In a LeRobot env: `python policy_bridge.py serve --checkpoint <unpacked>/pretrained_model --port 5555`
2. In the RoboTwin env: `python scripts/linux/twin_console.py --level 2 --executor policy`
The planner still splits commands into steps; each step is executed by the learned policy.
For SmolVLA add `--camera-map wrist=camera1,workspace=camera2` to the serve command.